# TCM-FuzzyWiki V5.0 · MiniMax-M3 Colab（多并发 + 断点续跑）

本 notebook 完整复现 `https://github.com/pariskang/TCM-FuzzyWiki`，并用 **MiniMax-M3** 做 observation-first 抽取。

特性：

- 支持 **OpenAI SDK 协议**（`/v1`）与 **Anthropic SDK 协议**（`/anthropic`）两种 MiniMax-M3 接入方式，`--provider` 一键切换。
- **多并发**抽取（`--workers`）。
- **chunk 级断点续跑**：Colab 断线/限流/崩溃后，重跑同一 cell 只补抽失败或缺失的 chunk；`partial_success` 来源不会被永久跳过。
- 抽取完成后复用仓库内**确定性 pipeline**：membership、共现、规则、Larsen 推理、Mamdani、三层聚合、关系网络、Markdown Wiki、validation、audit、manifest。
- observation ID 按输入顺序确定性分配，与并发完成顺序无关，**结果可复现**。

> 安全：**不要**把 API Key 写进 notebook。请用 Colab 左侧 🔑 Secrets 设置 `MINIMAX_API_KEY`，或运行时手动输入。


## 1. 挂载 Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆仓库并安装

克隆带 `build-llm` / `normalize-input` 命令的分支并以 editable 方式安装。


In [ ]:
%cd /content
!rm -rf TCM-FuzzyWiki
!git clone https://github.com/pariskang/TCM-FuzzyWiki.git
%cd /content/TCM-FuzzyWiki
!git checkout claude/elegant-mccarthy-p7zg2g || echo "（分支已合并入 main 时此步可忽略）"
!git rev-parse HEAD

# 核心依赖随包安装；额外装可选的 SDK 与 json-repair（增强容错）
!python -m pip install -q -e '.[dev]'
!python -m pip install -q --upgrade openai anthropic json-repair

# 先跑离线 demo，确认环境与确定性 pipeline 正常
!tcm-fuzzywiki run-demo --output /content/drive/MyDrive/TCM_FuzzyWiki_MiniMax_M3/demo
!python -m pytest -q

## 3. 配置 API Key 与运行参数

推荐在 Colab Secrets 里加 `MINIMAX_API_KEY` 并开启本 notebook 访问。


In [ ]:
import os, getpass, datetime
from pathlib import Path

key = None
try:
    from google.colab import userdata
    key = userdata.get('MINIMAX_API_KEY')
except Exception:
    key = None
key = key or os.environ.get('MINIMAX_API_KEY')
if not key:
    key = getpass.getpass('请输入 MiniMax API Key（不显示）：')
os.environ['MINIMAX_API_KEY'] = key
print('API Key loaded:', bool(os.environ.get('MINIMAX_API_KEY')))

# ===== 可调参数 =====
PROVIDER  = 'openai'      # 'openai'  -> https://api.minimaxi.com/v1
                         # 'anthropic'-> https://api.minimaxi.com/anthropic
MODEL     = 'MiniMax-M3'
WORKERS   = 6            # 并发；遇到 429/限流就调小（如 2-3）
CHUNK_CHARS = 2400      # 长章节切块大小
MAX_OBS_PER_CHUNK = 12
THINKING  = 'disabled'   # 结构化抽取建议关闭思考以稳定 JSON

# 输入古籍表格（任意列名，下一步会规范化）
RAW_INPUT = Path('/content/drive/MyDrive/中医古籍指令数据/book_chapters_split.xlsx')

RUN_TAG = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_ROOT = Path('/content/drive/MyDrive/TCM_FuzzyWiki_MiniMax_M3')
OUTPUT_DIR = DRIVE_ROOT / f'run_{RUN_TAG}'
NORMALIZED = OUTPUT_DIR / 'input' / 'chapters.normalized.xlsx'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_INPUT.exists(), f'找不到输入文件：{RAW_INPUT}'
print('OUTPUT_DIR:', OUTPUT_DIR)

## 4. 规范化任意表格为推荐字段

自动映射中/英文列名、猜测正文列、补默认 metadata，并输出可审计的列映射报告。


In [ ]:
!tcm-fuzzywiki normalize-input \
  --input "$RAW_INPUT" \
  --output "$NORMALIZED" \
  --report "$OUTPUT_DIR/input/column_mapping_report.json"

NORMALIZED_CSV = NORMALIZED.with_suffix('.csv')
print('规范化 CSV：', NORMALIZED_CSV)

import pandas as pd
display(pd.read_csv(NORMALIZED_CSV).head(3))

## 5. 配置/数据体检（doctor）


In [ ]:
!tcm-fuzzywiki doctor --config configs/tcm_fuzzywiki.yaml --input "$NORMALIZED_CSV" 

## 6. 小样本试跑（先验证 3 章）

确认 MiniMax 输出与字段映射无误，再跑全量。


In [ ]:
!tcm-fuzzywiki build-llm \
  --input "$NORMALIZED_CSV" \
  --config configs/tcm_fuzzywiki.yaml \
  --output "$OUTPUT_DIR" \
  --provider "$PROVIDER" \
  --model "$MODEL" \
  --workers 3 \
  --chunk-chars $CHUNK_CHARS \
  --max-observations-per-chunk $MAX_OBS_PER_CHUNK \
  --thinking "$THINKING" \
  --limit 3

!cat "$OUTPUT_DIR/summary.txt" 

## 7. 全量运行（多并发 + 断点续跑）

直接重跑本 cell 即可续跑：`success` 的 chunk 自动跳过，只补抽失败/缺失 chunk。
切勿在续跑时修改 `--chunk-chars`/`--chunk-overlap` 或更换输入文件（manifest 会拒绝错配）。


In [ ]:
!tcm-fuzzywiki build-llm \
  --input "$NORMALIZED_CSV" \
  --config configs/tcm_fuzzywiki.yaml \
  --output "$OUTPUT_DIR" \
  --provider "$PROVIDER" \
  --model "$MODEL" \
  --workers $WORKERS \
  --chunk-chars $CHUNK_CHARS \
  --max-observations-per-chunk $MAX_OBS_PER_CHUNK \
  --thinking "$THINKING"

!cat "$OUTPUT_DIR/summary.txt" 

## 8. 实时进度 / 断点状态

抽取过程中可随时运行查看进度；中断后用于确认断点。


In [ ]:
import pandas as pd
from pathlib import Path
ext = Path(OUTPUT_DIR) / 'extraction'
sp = ext / 'source_progress.csv'
print('--- live_status.txt ---')
print((ext / 'live_status.txt').read_text() if (ext/'live_status.txt').exists() else '尚未开始')
if sp.exists():
    df = pd.read_csv(sp)
    display(df['status'].value_counts(dropna=False))
    display(df[df['status'].isin(['error','partial_success'])].head(20))

## 9. 关键产物检查


In [ ]:
import pandas as pd, json
from pathlib import Path
data = Path(OUTPUT_DIR) / 'data'
for name in ['observations.csv','memberships.csv','candidate_patterns.csv','rules.csv',
             'inference_results.csv','aggregation_results.csv','relation_edges.csv',
             'validation_report.csv','completion_assessment.csv','run_manifest.json']:
    p = data / name
    print(('OK  ' if p.exists() else 'MISS'), name)

display(pd.read_csv(data/'observations.csv').head(10))
display(pd.read_csv(data/'completion_assessment.csv'))
man = json.loads((data/'run_manifest.json').read_text())
print('provider:', man.get('llm_provider'), '| model:', man.get('llm_model'))
print('extraction_report:', json.dumps(man.get('extraction_report', {}), ensure_ascii=False))

## 10. 评估金标准 / 专家审核（可选）

- 把专家金标准 CSV 放到一个目录，运行时加 `--gold-dir path/to/gold` 即可计算 FCR/CRP/MIC/SMB/FIA。
- 专家审核 `data/expert_rule_review.csv` 后整理为 `rules.csv`，加 `--rules-csv path/to/rules.csv` 重跑升格正式规则。
- `wiki/index.md` 为生成的 Markdown Wiki 首页；`wiki/audit/*.md` 为审计、manifest、validation、completion 报告。

> 再次提醒：请勿把 API Key 写入 notebook 或提交到仓库。
